# Replenishment and model-based tail correction

Births compete with market consumption and state-dependent cancellation. A residual-time correction uses the known model; it is not an uncensored empirical estimate.

Synthetic data only. All settings and random seeds are specified below.

In [1]:
from pathlib import Path
import sys, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "results" / "synthetic" / "queues"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": .2})
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.0', 'numpy': '1.26.0', 'pandas': '2.1.1'}


In [2]:
from queue_models.replenishment import Config,simulate,mean_reference,corrected_mean
from queue_models.queue_depletion import summarize
rows=[]
for arrival in (0.,1.,2.,3.):
    c=Config(3,arrival,1.,.2)
    sample=simulate(c,[100.],samples=5000,seed=410)[0]
    rows.append(dict(arrival=arrival,**summarize(sample,100.),**corrected_mean(c,sample),unrestricted_theory=mean_reference(c)["mean"]))
table=pd.DataFrame(rows);display(table)
table.to_csv(OUT/"replenishment.csv",index=False)
for key,label in [("restricted_mean","Restricted mean"),("corrected_mean","Model-corrected mean"),("unrestricted_theory","Unrestricted theory")]:
    plt.plot(table.arrival,table[key],"o-",label=label)
plt.xlabel("Arrival rate");plt.ylabel("Time (model units)");plt.legend()
plt.tight_layout();plt.savefig(OUT/"replenishment.png",dpi=140);plt.show()


,arrival,cutoff,samples,depleted,censored,survival,survival_low,survival_high,restricted_mean,restricted_se,completed_only_mean,corrected_mean,corrected_se,correction_tail_bound,unrestricted_theory
0,0.0,100.0,5000,5000,0,0.000,0.000000,0.003190,2.202723,0.017946,2.202723,2.202723,0.017946,0.000000e+00,2.172619
1,1.0,100.0,5000,5000,0,0.000,0.000000,0.003190,5.117805,0.068713,5.117805,5.117805,0.068713,0.000000e+00,5.092021
2,2.0,100.0,5000,4895,105,0.021,0.014287,0.030769,22.646713,0.349235,20.987449,23.227775,0.378471,7.018054e-11,23.555145
3,3.0,100.0,5000,1840,3160,0.632,0.604338,0.658819,75.488191,0.530350,33.391823,272.430561,2.597721,6.529100e-11,271.631151


/var/folders/p2/_bpfxjzn1l301l3jgtn6bgx80000gn/T/ipykernel_57335/2886357194.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig(OUT/"replenishment.png",dpi=140);plt.show()


## Interpretation

Compare the displayed model reference with the simulation. Finite-sample agreement is an implementation check, not evidence of real-market calibration. See [assumptions](../../../docs/model_assumptions.md) and [limitations](../../../docs/validation_and_limitations.md).